In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │ Cell 1: Preprocess & Per-User 70/30 Train-Test Split                  │
# └────────────────────────────────────────────────────────────────────────┘

import pandas as pd
import numpy as np

# 1) Load & parse timestamps
df = pd.read_json('hackathon_2024/hackathon_data/transactions.json')
df['ts'] = pd.to_datetime(df['updated_timestamp'], errors='coerce')

# Fill missing timestamps per-user
df = df.sort_values(['owner_user_id','ts'])
df['ts'] = df.groupby('owner_user_id')['ts'] \
             .transform(lambda s: s.fillna(method='ffill').fillna(method='bfill'))

# 2) Numeric geo, forward/back-fill then 0
for col in ['geolocation_latitude','geolocation_longitude']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df.groupby('owner_user_id')[col] \
                .transform(lambda s: s.fillna(method='ffill')
                                    .fillna(method='bfill')
                                    .fillna(0))

# 3) Lagged coords for distance
df[['prev_lat','prev_lon']] = (
    df.groupby('owner_user_id')[
      ['geolocation_latitude','geolocation_longitude']
    ]
    .shift(1)
    .fillna(method='ffill')
    .fillna(method='bfill')
    .fillna(0)
)

# 4) Haversine distance
def hav(lat1, lon1, lat2, lon2):
    R = 6371.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (np.sin(dlat/2)**2 +
         np.cos(np.radians(lat1))*np.cos(np.radians(lat2))*np.sin(dlon/2)**2)
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['dist_from_last'] = hav(
    df['prev_lat'], df['prev_lon'],
    df['geolocation_latitude'], df['geolocation_longitude']
)

# 5) Velocity per hour
df = df.sort_values(['owner_user_id','ts'])
def compute_vel(g):
    return (g.set_index('ts')['event_id']
              .rolling('1h').count()
              .subtract(1)
              .fillna(0)
              .astype(int)
              .reset_index(drop=True))
df['tx_count_last_hour'] = df.groupby('owner_user_id').apply(compute_vel).explode().astype(int).values

# 6) Other features
df['cat_freq']    = df.groupby(['owner_user_id','merchant_category_code'])['event_id'].transform('count')
df['hour']        = df['ts'].dt.hour.astype(int)
df['day_of_week'] = df['ts'].dt.dayofweek.astype(int)
df['amount_z']    = df.groupby('owner_user_id')['amount'] \
                     .transform(lambda x: (x.fillna(x.mean()) - x.mean())/x.std() if x.std() else 0)

# 7) Per-user 70/30 split
FEATURES = ['amount_z','hour','day_of_week','dist_from_last','tx_count_last_hour','cat_freq']
user_splits = {}
for uid, grp in df.groupby('owner_user_id'):
    X = grp[FEATURES].fillna(0).values.astype(np.float32)
    n = len(X)
    if n < 10: 
        continue
    k = int(0.7 * n)
    user_splits[uid] = (X[:k], X[k:])

print("Users ready:", list(user_splits.keys())[:5])


Users ready: [102, 137, 207, 282, 334]


C:\Users\Windows 11 ENG\AppData\Local\Temp\ipykernel_40432\1601648828.py:15: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .transform(lambda s: s.fillna(method='ffill').fillna(method='bfill'))
C:\Users\Windows 11 ENG\AppData\Local\Temp\ipykernel_40432\1601648828.py:21: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .transform(lambda s: s.fillna(method='ffill')
C:\Users\Windows 11 ENG\AppData\Local\Temp\ipykernel_40432\1601648828.py:22: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(method='bfill')
C:\Users\Windows 11 ENG\AppData\Local\Temp\ipykernel_40432\1601648828.py:31: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .fillna(m

In [ ]:
# ┌────────────────────────────────────────────────────────────────────────┐
# │ Cell 2: Train & Evaluate Autoencoder for One Demo User               │
# └────────────────────────────────────────────────────────────────────────┘

import torch
import torch.nn as nn

# 1) Pick a demo user
uid, (X_train, X_test) = next(iter(user_splits.items()))
print(f"Demo user_id = {uid}, train size = {len(X_train)}, test size = {len(X_test)}")

# 2) Define the autoencoder
class AE(nn.Module):
    def __init__(self, D, H=3):
        super().__init__()
        self.enc = nn.Linear(D, H)
        self.dec = nn.Linear(H, D)
    def forward(self, x):
        return self.dec(self.enc(x))

model     = AE(X_train.shape[1], H=3)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# 3) Training loop
model.train()
xt = torch.from_numpy(X_train)
for _ in range(100000):
    recon = model(xt)
    loss = criterion(recon, xt)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"✅ Train MSE = {loss.item():.4f}")

# 4) Evaluate on the test set
model.eval()
with torch.no_grad():
    xts       = torch.from_numpy(X_test)
    test_loss = criterion(model(xts), xts).item()
print(f"📊 Test MSE = {test_loss:.4f}")


Demo user_id = 102, train size = 702, test size = 301


In [ ]:
model.train()
xt = torch.from_numpy(X_train)
for _ in range(50000):
    recon = model(xt)
    loss = criterion(recon, xt)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
print(f"✅ Train MSE = {loss.item():.4f}")


✅ Train MSE = 0.9105


In [43]:
torch.save(model.state_dict(), "autoencoder.pth")


In [26]:
model.eval()
with torch.no_grad():
    xts       = torch.from_numpy(X_test)
    test_loss = criterion(model(xts), xts).item()
print(f"📊 Test MSE = {test_loss:.4f}")


📊 Test MSE = 0.6636


In [29]:
X_all = df[FEATURES].fillna(0).values.astype(np.float32)
model.eval()
with torch.no_grad():
    recon_all = model(torch.from_numpy(X_all)).cpu().numpy()

# 3) Compute the mean‐squared reconstruction error per row
ae_errors = np.mean((X_all - recon_all)**2, axis=1)
df['ae_error'] = ae_errors


In [30]:
train_cut = int(0.7 * n)
threshold = np.percentile(ae_errors[:train_cut], 97.5)

# 5) Flag anomalies
df['ae_anomaly'] = (df['ae_error'] > threshold).astype(int)

In [34]:
df.columns

Index(['event_id', 'transaction_description', 'transaction_type', 'iban',
       'counterparty_name', 'counterparty_iban', 'amount', 'currency',
       'transaction_subtype', 'monetary_account_id', 'updated_timestamp',
       'place_name', 'place_address', 'category', 'geolocation_latitude',
       'geolocation_longitude', 'geolocation_altitude', 'geolocation_radius',
       'user_type', 'merchant_category_code', 'tags', 'opening_periods',
       'number_of_recommendation_total', 'recommendations_category',
       'categories_friends', 'url_google_maps', 'city', 'country',
       'event_type', 'inner_id', 'owner_user_id', '.transaction_subtype',
       'number_of_recommendations_total', 'transaction_subnetype',
       'counterparty_ibank', 'eventType', 'counterparty_ibn', 'ts', 'prev_lat',
       'prev_lon', 'dist_from_last', 'tx_count_last_hour', 'cat_freq', 'hour',
       'day_of_week', 'amount_z', 'ae_error', 'ae_anomaly'],
      dtype='object')

In [37]:
import numpy as np

# 1) Reconstruct & compute per‐feature squared errors
X_all      = df[FEATURES].fillna(0).values.astype(np.float32)
with torch.no_grad():
    recon_all = model(torch.from_numpy(X_all)).cpu().numpy()

feat_err   = (X_all - recon_all)**2
# normalize so different scales are comparable
norm_err   = feat_err / (feat_err.mean(axis=0) + 1e-6)

# 2) Map feature names to human reasons
FEATURE_NAMES = FEATURES
reason_map = {
    'amount_z'          : 'unusual transaction size',
    'hour'              : 'atypical time of day',
    'day_of_week'       : 'odd day of week',
    'dist_from_last'    : 'unexpected location jump',
    'tx_count_last_hour': 'spike in transaction velocity',
    'cat_freq'          : 'rare merchant category'
}

def top_reasons(err_row, top_n=2):
    # pick top_n features by normalized error
    idxs = np.argsort(err_row)[-top_n:][::-1]
    return [ FEATURE_NAMES[i] for i in idxs ]

def human_reasons(err_row, top_n=2):
    keys = top_reasons(err_row, top_n)
    return "; ".join(reason_map[k] for k in keys)

# 3) Attach to df
df['ae_top_features'] = [ top_reasons(row) for row in norm_err ]
df['ae_reasons']      = [ human_reasons(row) for row in norm_err ]

# 4) Now select anomalies with reasons
anoms_only = df.loc[
    df['ae_anomaly'] == 1,
    [
        'owner_user_id','event_id','ts',
        'amount','transaction_type','counterparty_name','merchant_category_code','country',
        'amount_z','hour','day_of_week','dist_from_last','tx_count_last_hour','cat_freq',
        'ae_error','ae_anomaly','ae_top_features','ae_reasons'
    ]
]

anoms_only


,owner_user_id,event_id,ts,amount,transaction_type,counterparty_name,merchant_category_code,country,amount_z,hour,day_of_week,dist_from_last,tx_count_last_hour,cat_freq,ae_error,ae_anomaly,ae_top_features,ae_reasons
8665,102,248508973,2019-08-04 10:45:13.605406,-23.57,FIS,Lidl,5411,,0.041668,10,6,7420.000399,0,56.0,2.070114,1,"[cat_freq, dist_from_last]",rare merchant category; unexpected location jump
1783,102,48952870,2019-10-20 15:37:32.820479,-45.19,IDEAL,Jumbo,5412,,0.040597,15,6,7420.500399,0,5.0,1.941251,1,"[cat_freq, dist_from_last]",rare merchant category; unexpected location jump
3747,102,491596523,2020-05-18 15:34:27.438571,-9.99,BUNQ,Spotify * Subscription,5812,,0.042341,15,0,0.000000,2,85.0,1.923097,1,"[tx_count_last_hour, day_of_week]",spike in transaction velocity; odd day of week
6711,102,25639852,2020-08-16 12:33:57.799938,10.50,MASTERCARD,Clara,5814,USA,0.043355,12,6,5582.972523,0,60.0,1.949056,1,"[cat_freq, hour]",rare merchant category; atypical time of day
9672,102,778954612,2021-02-10 09:23:36.137596,-630000.00,PAYMENT,NOUVEL PERSPECTIVE SA,,,-31.161893,9,2,0.000000,0,484.0,160.666794,1,"[amount_z, hour]",unusual transaction size; atypical time of day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9109,771,2034698550,2024-06-02 11:34:20.164572,-8.73,PAYMENT,AB *JAVA CAFFEEN,5411,US,0.013671,11,6,10481.149866,0,52.0,2.165217,1,"[cat_freq, dist_from_last]",rare merchant category; unexpected location jump
4040,771,2365909537,2024-06-16 14:40:12.123456,-185.75,IDEAL,Breeze Cafe House,5814,US,-0.025202,14,6,8580.213201,0,63.0,2.015929,1,"[cat_freq, dist_from_last]",rare merchant category; unexpected location jump
6087,771,8326952852,2024-07-21 11:32:43.985932,-75.66,SWIFT,Lucio's Bakery,,CA,-0.001026,11,6,5965.591921,0,490.0,1.986512,1,"[cat_freq, hour]",rare merchant category; atypical time of day
6165,771,712345678,2024-12-15 02:50:45.452000,-40.40,IDEAL,CNM TRVL AGENCY,,Netherlands,0.006717,2,6,8943.316742,0,490.0,2.348548,1,"[cat_freq, dist_from_last]",rare merchant category; unexpected location jump


In [ ]:
anoms_only = df.loc[
    df['ae_anomaly'] == 1,
    [
        # identifiers & timestamp
        'owner_user_id',
        'ts',

        # raw transaction context
        'amount',
        'transaction_type',
        'counterparty_name',
        'country',

        # engineered features
        'hour',
        'day_of_week',
        
        # autoencoder diagnostics
        'ae_error',
        'ae_anomaly',
        'ae_reasons'
    ]
]

anoms_only


,owner_user_id,ts,amount,transaction_type,counterparty_name,country,hour,day_of_week,ae_error,ae_anomaly,ae_reasons
8665,102,2019-08-04 10:45:13.605406,-23.57,FIS,Lidl,,10,6,2.070114,1,rare merchant category; unexpected location jump
1783,102,2019-10-20 15:37:32.820479,-45.19,IDEAL,Jumbo,,15,6,1.941251,1,rare merchant category; unexpected location jump
3747,102,2020-05-18 15:34:27.438571,-9.99,BUNQ,Spotify * Subscription,,15,0,1.923097,1,spike in transaction velocity; odd day of week
6711,102,2020-08-16 12:33:57.799938,10.50,MASTERCARD,Clara,USA,12,6,1.949056,1,rare merchant category; atypical time of day
9672,102,2021-02-10 09:23:36.137596,-630000.00,PAYMENT,NOUVEL PERSPECTIVE SA,,9,2,160.666794,1,unusual transaction size; atypical time of day
...,...,...,...,...,...,...,...,...,...,...,...
9109,771,2024-06-02 11:34:20.164572,-8.73,PAYMENT,AB *JAVA CAFFEEN,US,11,6,2.165217,1,rare merchant category; unexpected location jump
4040,771,2024-06-16 14:40:12.123456,-185.75,IDEAL,Breeze Cafe House,US,14,6,2.015929,1,rare merchant category; unexpected location jump
6087,771,2024-07-21 11:32:43.985932,-75.66,SWIFT,Lucio's Bakery,CA,11,6,1.986512,1,rare merchant category; atypical time of day
6165,771,2024-12-15 02:50:45.452000,-40.40,IDEAL,CNM TRVL AGENCY,Netherlands,2,6,2.348548,1,rare merchant category; unexpected location jump


In [41]:
# Add tiered action based on error levels

import numpy as np

# 1) Compute thresholds on training errors
split = int(0.7 * len(X))    # 70% of total rows, rounded down
errors_train = ae_errors[:split]  # first 70% were used for training
warn_thresh = np.percentile(errors_train, 97.5)   # warning threshold
fa_thresh   = np.percentile(errors_train, 99.5)   # stricter threshold for 2FA

# 2) Define action column
def decide_action(err, warn_t=warn_thresh, fa_t=fa_thresh):
    if err > fa_t:
        return '2FA_required'
    elif err > warn_t:
        return 'warn'
    else:
        return 'none'

df['ae_action'] = df['ae_error'].apply(decide_action)

# 3) Select only flagged transactions
flagged = df[df['ae_action'] != 'none']
cols = [
    'owner_user_id','event_id','ts',
    'amount','transaction_type','counterparty_name',
    'ae_error','ae_action','ae_reasons'
]


In [42]:
anoms_only = df.loc[
    df['ae_action'] == "2FA_required",
    [
        # identifiers & timestamp
        'owner_user_id',
        'ts',

        # raw transaction context
        'amount',
        'transaction_type',
        'counterparty_name',
        'country',

        # engineered features
        'hour',
        'day_of_week',
        
        # autoencoder diagnostics
        'ae_error',
        'ae_anomaly',
        'ae_reasons'
    ]
]

anoms_only


,owner_user_id,ts,amount,transaction_type,counterparty_name,country,hour,day_of_week,ae_error,ae_anomaly,ae_reasons
8665,102,2019-08-04 10:45:13.605406,-23.57,FIS,Lidl,,10,6,2.070114,1,rare merchant category; unexpected location jump
9672,102,2021-02-10 09:23:36.137596,-630000.00,PAYMENT,NOUVEL PERSPECTIVE SA,,9,2,160.666794,1,unusual transaction size; atypical time of day
3261,102,2021-05-02 10:22:15.521978,-45.75,WISE,Texas Electronics,,10,6,2.090993,1,rare merchant category; unexpected location jump
2378,102,2022-04-10 15:20:57.934128,-509.52,BUNQ,SO *MILLION STARS CO,,15,6,2.269056,1,spike in transaction velocity; atypical time o...
5511,137,2020-05-17 08:14:15.912315,-46.25,PAYMENT,Blues Cafe,,8,6,2.471660,1,rare merchant category; unexpected location jump
...,...,...,...,...,...,...,...,...,...,...,...
834,771,2023-08-13 10:21:13.677045,-153.39,PAYMENT,THE CAPITOL HOTEL TOKYU,JP,10,6,2.144982,1,rare merchant category; unexpected location jump
1039,771,2024-01-10 14:07:35.613948,123482.00,FIS,TAXAUTHORITY,,14,2,123.242897,1,unusual transaction size; atypical time of day
9109,771,2024-06-02 11:34:20.164572,-8.73,PAYMENT,AB *JAVA CAFFEEN,US,11,6,2.165217,1,rare merchant category; unexpected location jump
6165,771,2024-12-15 02:50:45.452000,-40.40,IDEAL,CNM TRVL AGENCY,Netherlands,2,6,2.348548,1,rare merchant category; unexpected location jump


In [49]:
import pandas as pd
import random
# 1) Define how many samples per action type
sample_counts = {'none': 30, 'warn':20, '2FA_required': 20}

# 2) Collect samples by action
frames = []
for action, count in sample_counts.items():
    subset = df[df['ae_action'] == action]
    n = min(count, len(subset))
    frames.append(subset.sample(n, random_state=42))

sample_df = pd.concat(frames).reset_index(drop=True)
random.shuffle(sample_df.values)
# 3) List only the original “raw” columns to keep
raw_cols = [
    'event_id','transaction_description','transaction_type','iban',
    'counterparty_name','counterparty_iban','amount','currency',
    'transaction_subtype','monetary_account_id','updated_timestamp',
    'place_name','place_address','category','geolocation_latitude',
    'geolocation_longitude','geolocation_altitude','geolocation_radius',
    'user_type','merchant_category_code','tags','opening_periods',
    'number_of_recommendation_total','recommendations_category',
    'categories_friends','url_google_maps','city','country',
    'event_type','inner_id','owner_user_id','.transaction_subtype',
    'number_of_recommendations_total','transaction_subnetype',
    'counterparty_ibank','eventType','counterparty_ibn'
]

# 4) Subset to only raw data
raw_sample = sample_df[raw_cols]

# 5) Save to CSV
raw_sample.to_csv('sample_raw_transactions.csv', index=False)
print(f"Saved {len(raw_sample)} raw transactions to sample_raw_transactions.csv")


Saved 70 raw transactions to sample_raw_transactions.csv


In [46]:
# 1) Wrap your training array in a DataFrame
df_Xtrain = pd.DataFrame(X_train, columns=FEATURES)

# 2) (Optional) if you want to preserve the original row‐indices from df:
#    df_Xtrain.index = df[df['owner_user_id']==demo_uid].index[:len(X_train)]

# 3) Write to CSV
df_Xtrain.to_csv('X_train.csv', index=False)


In [ ]:
7